# calibration-repro: CIFAR-100 ResNet training

Guo et al. (2017), *On Calibration of Modern Neural Networks*.

Kaggle: GPU T4 x2, Internet on, Save Version -> Save & Run All. Set `PART` (1-4) below; each part fits in one
12 h session. Download `logits_part<PART>.zip` from the Output tab.

In [ ]:
import os, subprocess
REPO = "https://github.com/BenarmasNADIR/calibration-repro.git"
if not os.path.exists("calibration-repro"):
    subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
os.chdir("calibration-repro")
print(os.getcwd())

In [ ]:
import importlib.util, re
need = {"torch": "torch", "torchvision": "torchvision", "numpy": "numpy", "yaml": "pyyaml"}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
pins = [l.strip() for l in open("requirements.txt") if re.match(r"^(%s)==" % "|".join(missing), l)] if missing else []
print("missing:", missing or "none")
if pins:
    subprocess.run(["pip", "install", "-q", *pins], check=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!python -c "import sys; sys.path.insert(0, 'src'); from data import load_cifar100; load_cifar100('data')"

In [ ]:
!pip install -q pytest
!python src/train.py --config configs/smoke.yaml --device cpu
!python -m pytest -q

In [ ]:
PART = "1"
WDS = ["1e-5", "1e-4", "5e-4", "1e-3", "5e-3"]
r110 = lambda seeds: [f"--config configs/resnet110_main.yaml --seed {s}" for s in seeds]
r56_wd = lambda s: [f"--config configs/wd_ablation.yaml --wd {wd} --seed {s}" for wd in WDS]
depth = lambda s: [f"--config configs/depth_ablation.yaml --depth {d} --seed {s}" for d in [20, 32]]
PARTS = {
    "1": r110([0, 1]),
    "2": r110([2]) + depth(0) + depth(1),
    "3": r56_wd(0),
    "4": r56_wd(1),
}
PARTS["all"] = [j for k in "1234" for j in PARTS[k]]
JOBS = PARTS[PART]
print(f"part {PART}: {len(JOBS)} jobs")

In [ ]:
import torch, time, pathlib
pathlib.Path("results/logs").mkdir(parents=True, exist_ok=True)
n_gpu = max(torch.cuda.device_count(), 1)
queue, running = list(JOBS), {}
while queue or running:
    for gpu in range(n_gpu):
        if gpu not in running and queue:
            job = queue.pop(0)
            log = open("results/logs/" + job.replace("--", "").replace("configs/", "").replace(" ", "_") + ".log", "a")
            env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
            running[gpu] = (subprocess.Popen(["python", "src/train.py", *job.split()],
                                             stdout=log, stderr=subprocess.STDOUT, env=env), job)
            print(f"GPU {gpu}: started {job}")
    for gpu, (p, job) in list(running.items()):
        if p.poll() is not None:
            print(f"GPU {gpu}: finished {job} (exit {p.returncode})")
            del running[gpu]
    time.sleep(30)
!tail -n 2 results/logs/*.log

In [ ]:
ZIP = f"logits_part{PART}.zip"
!cd results && zip -qr ../../{ZIP} logits logs && ls -la ../../{ZIP}
try:
    from google.colab import files
    files.download(f"../{ZIP}")
except ImportError:
    pass

Locally: unzip each `logits_part*.zip` into `results/`, then `uv run python src/analyze.py`.